# COMP3991 Lab 3 — Applications of differentiation; multivariable differentiation

**Week 3 · 2 hours · follows Lecture 3**

Notation and laws come from Cohen, *Calculus Unraveled*. The lecture used the same symbols; this lab does too.

| Item | Detail |
|---|---|
| Syllabus | 2.6 Newton, bisection, gradient descent; 3.5–3.7 two-variable functions, partials, max/min |
| Cohen | Ch. 9 (pp. 287–307); Ch. 10 (pp. 314–340); §6.3.3 (pp. 206–207); critical points §§6.1–6.2 |
| Companion notebooks | [ch09_applications_exercises.ipynb](https://github.com/mikexcohen/Calculus_book/blob/main/exercises/ch09_applications_exercises.ipynb) · [ch10_multivariableDiff_exercises.ipynb](https://github.com/mikexcohen/Calculus_book/blob/main/exercises/ch10_multivariableDiff_exercises.ipynb) · [ch06_criticalPoints_exercises.ipynb](https://github.com/mikexcohen/Calculus_book/blob/main/exercises/ch06_criticalPoints_exercises.ipynb) |
| YouTube | [Calculus Unraveled playlist, Ch. 6, 9, and 10](https://www.youtube.com/playlist?list=PLn0OLiymPak2TFbFj2SD6-p-3-jOah7fe) |

Run this notebook in **Google Colab** or locally with `numpy`, `sympy`, and `matplotlib`.

**How this notebook is organised.** Each task has **questions**, a **templated code cell** (complete every `TODO` — running the cell as-is will fail), written answers, then **Solutions (tutor)** with the worked algebra *and* a separate complete-code cell. **Task 9 is for credit (20 marks).** Tasks 2–4 are the rehearsal for it (Newton, bisection, GD, Newton-on-$C'$ vs Newton-on-$C$). Tutors: delete or hide every cell whose heading starts with **Solutions (tutor)** or **Mark scheme (tutor)** before distributing.

**Suggested timing**

| Minutes | Task |
|---|---|
| 0–8 | Setup and notation |
| 8–20 | Task 1 Linear approximation |
| 20–38 | Task 2 Newton’s method |
| 38–52 | Task 3 Bisection (syllabus patch) |
| 52–70 | Task 4 Critical points and 1D gradient descent |
| 70–82 | Task 5 Multivariable limits |
| 82–95 | Task 6 Partials and Clairaut |
| 95–108 | Task 7 Gradient, Hessian, 2D GD |
| 108–120 | Task 8 Model: two weights |
| After the 2-hour lab | **Task 9 (for credit)** logistic neuron + Newton / GD / bisection |


## Notation used in this lab

Every abbreviation below is defined here and then used without re-explaining it.

| Symbol / phrase | Cohen | Meaning |
|---|---|---|
| **d.n.e.** | §4.5, §10.2 | **d**oes **n**ot **e**xist |
| **c.p.** | §6.1 | **c**ritical **p**oint |
| $x_s$, $x_a$ | §9.1 | starting point; point being approximated |
| $L(x_a)$ | (9.10) | linear approximation $f(x_s)+f'(x_s)(x_a-x_s)$ |
| $x_n$ | (9.23) | Newton iterate |
| $\eta$ | (6.8), §10.6 | **learning rate** (small positive scalar) |
| $\partial$, $f_x$, $f_y$ | (10.16)–(10.17) | del / partial derivatives |
| $\nabla f$ | (10.43) | gradient (nabla / “del”) |
| $\mathbf{J}$ | §10.5.3 | Jacobian: rows = functions, columns = variables |
| $H_f$ | (10.54) | Hessian of second partials |
| $z_0=(x_0,y_0)$ | (10.64) | current 2D guess in gradient descent |
| $\hat{y}=wx$ | Labs 1–2 | prediction: **weight** times data |
| $C$ | Labs 1–2 | squared-error **cost** (not a weight) |
| $\sigma(u)=1/(1+e^{-u})$ | Lab 2 Task 6 | logistic / sigmoid; range $(0,1)$ |
| $y=0.8$ | Lab 2 Task 6C | target probability (not Lab 1's $y=2$) |

**Convention.** A two-sided limit (1D) or a path-independent limit (2D) must exist before we say the limit exists. If paths disagree, write **d.n.e.** Never write an undefined “DNE”.


## Task 0 — Setup (questions)

Run the next cell. If you are on Colab and a package is missing, uncomment the `pip` line.

**0a.** In Cohen’s language, is $\eta$ a weight, a learning rate, or a predicted output?

**0b.** Newton (9.23) solves $f(x)=0$. Gradient descent (6.8) does what to $f$?


In [ ]:
# %pip install numpy sympy matplotlib

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

np.set_printoptions(precision=8, suppress=True)
sp.init_printing()

x, y = sp.symbols("x y", real=True)
w, w1, w2 = sp.symbols("w w1 w2", real=True)
print("numpy", np.__version__, "| sympy", sp.__version__)


numpy 2.1.3 | sympy 1.14.0


**Your answers (Task 0)**

- 0a: η (eta) represents the learning rate, which determines how much the model changes its weights during each step of gradient descent.
- 0b: Newton’s method finds the roots of \(f(x)\), while gradient descent tries to find the minimum value of \(f(x)\).


## Task 1 — Linear approximation (questions)

Cohen §9.1. Starting point $x_s$, approximation point $x_a$:

$$L(x_a)=f(x_s)+f'(x_s)\,(x_a-x_s)\qquad\text{(9.10)}$$

Cohen’s worked function:

$$f(x)=\sqrt{x}+2x,\qquad x_s=4,\qquad x_a=4.37.\qquad\text{(9.2)}$$

**1a.** By hand, compute $f(x_s)$, $f'(x_s)$, and $L(x_a)$.

**1b.** Implement (9.10) in the next cell. Compare $L(x_a)$ to `np.sqrt(4.37)+2*4.37`. Is the error “small”? What would you expect if $x_a=9$ with the same $x_s=4$?

**1c.** For $g(\zeta)=\lvert\sin(\zeta)\rvert$, $x_s=\pi/2$, why is $L(x_a)$ the same number for every $x_a$? (Cohen (9.16)–(9.21).)


In [ ]:
# Task 1 — complete the TODOs. This cell fails until you do.

def linear_approx(f, fp, xs, xa):
    return f(xs) + fp(xs) * (xa - xs)    #Cohen (9.10)

def f_cohen(t):
    return np.sqrt(t) + 2 * t


def fp_cohen(t):
    return 0.5 / np.sqrt(t) + 2.0


xs, xa = 4.0, 4.37
L_code = linear_approx(f_cohen, fp_cohen, xs, xa)
L_hand = 10.8325  # TODO: your value from 1a
if L_hand is None:
    raise NotImplementedError("Task 1b: set L_hand from 1a")
f_true = f_cohen(xa)
print("L_code     =", L_code)
print("L_hand     =", L_hand)
print("true f(xa) =", f_true)
print("|L - f|    =", abs(L_code - f_true))
print("if xa=9, L =", linear_approx(f_cohen, fp_cohen, xs, 9.0), " true =", f_cohen(9.0))


L_code     = 10.8325
L_hand     = 10.8325
true f(xa) = 10.830454496036687
|L - f|    = 0.0020455039633127114
if xa=9, L = 21.25  true = 21.0


**Your answers (Task 1)**

- 1a:

f'(x) = 1/(2x) + 2

f'(4) = 1/4 + 2 = 2.25

L(4.37) = 10 + 2.25(0.37) = 10.8325
- 1b:

L(4.37) = 10.8325, the true value is approximately 10.83045, giving an error of about 0.002.

If xa = 9, then L(9) = 21.25, the true value is 21, so the error is 0.25. This is a bigger error because 9 is farther from 4. The farther xa is from xs, the less accurate the approximation.

- 1c:

g'(π/2) = sgn(sin(π/2)) cos(π/2) = 1 × 0 = 0

Since the slope is 0, the linear approximation is:

L(xa) = 1 + 0 × (xa − π/2) = 1

This means the approximation stays at 1 for every xa because the tangent line is flat.


## Task 2 — Newton’s method (questions)

Cohen §9.2. Root means $f(x)=0$. Update:

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}\qquad\text{(9.23)}$$

Cohen’s cubic, rewritten as $f(x)=0$:

$$f(x)=3x^{3}+2x-7\pi\qquad\text{(9.22)}$$

You will **reuse** `newton_step` in Task 4 and Task 9. If $f'(x_n)=0$, the update is **d.n.e.** (**d**oes **n**ot **e**xist; Cohen §9.2.2). Return `np.nan` in that case. Never write “DNE”.

**2a.** Implement (9.23). Start at $x_0=1$ (Cohen’s figure). Print $x_0$ through $x_5$ and $f(x_n)$. Does $\lvert f(x_n)\rvert$ shrink?

**2b.** By hand, one Newton step for $f(x)=x^{2}-2$ from $x_0=1$. Then check with your function.

**2c.** What happens if you start Newton for $x^{2}-2$ at $x_0=0$? Write **d.n.e.** if the update is undefined, and say why.


In [ ]:
# Task 2 — complete the TODOs. This cell fails until you do.

def newton_step(f, fp, xn):
    if fp(xn) == 0:
       return np.nan
    return xn - f(xn) / fp(xn)


def newton_iter(f, fp, x0, n_steps=5):
    xs = [float(x0)]
    for _ in range(n_steps):
        xs.append(float(newton_step(f, fp, xs[-1])))
    return np.array(xs)


# 2a — Cohen (9.22)
f_cubic = lambda t: 3 * t**3 + 2 * t - 7 * np.pi   # TODO: lambda t: 3 * t**3 + 2 * t - 7 * np.pi
fp_cubic = lambda t: 9 * t**2 + 2  # TODO: lambda t: 9 * t**2 + 2
if f_cubic is None or fp_cubic is None:
    raise NotImplementedError("Task 2a: set f_cubic and fp_cubic")

xs_cubic = newton_iter(f_cubic, fp_cubic, 1.0, n_steps=5)
print("n        x_n           f(x_n)")
for n, xn in enumerate(xs_cubic):
    print(f"{n:d}  {xn:14.8f}  {f_cubic(xn):14.8f}")

# 2b
f_sq = lambda t: t**2 - 2   # TODO: lambda t: t**2 - 2
fp_sq = lambda t: 2 * t  # TODO: lambda t: 2 * t
if f_sq is None or fp_sq is None:
    raise NotImplementedError("Task 2b: set f_sq and fp_sq")
print("\n2b  x0=1 then one step:", newton_iter(f_sq, fp_sq, 1.0, n_steps=1))

# 2c
print("2c  x0=0:", newton_step(f_sq, fp_sq, 0.0))


n        x_n           f(x_n)
0      1.00000000    -16.99114858
1      2.54464987     32.52982967
2      2.00497920      6.19850771
3      1.84262737      0.46278944
4      1.82841283      0.00334216
5      1.82830868      0.00000018

2b  x0=1 then one step: [1.  1.5]
2c  x0=0: nan


**Your answers (Task 2)**

- 2a:

x₀ = 1

x₁ = 2.54465

x₂ = 2.00499

x₃ = 1.84262

x₄ = 1.82841

x₅ = 1.82830

Yes, |f(xₙ)| gets smaller with each iteration: 16.99, 32.53, 6.20, 0.46, 0.0035, and approximately 0. Newton's method is converging to the root, which is approximately 1.8283.

- 2b:

f(1) = −1

f'(1) = 2

x₁ = 1 − (−1)/2 = 1.5

- 2c:

At x₀ = 0, f'(0) = 0, so the Newton update would involve division by zero. Therefore, the update is undefined, and we return np.nan.


## Task 3 — Bisection (questions)

**Not in Cohen.** Syllabus 2.6; lecture patch from continuity.

If $f$ is continuous on $[a,b]$ and $f(a)$ and $f(b)$ have opposite signs, then $f$ has a root in $(a,b)$. Midpoint $m=(a+b)/2$; keep the half-interval that still has a sign change.

Use the same $f$ as Task 2a (Cohen’s cubic). You will **reuse** `bisection` in Task 4 (on $C'$) and Task 9 (on the logistic $C'$).

**3a.** Evaluate $f(1)$ and $f(2)$. Do they have opposite signs? What does continuity then guarantee?

**3b.** Complete `bisection` so it returns the midpoints after `n_steps` halvings (or earlier if $f(m)=0$). Run it on $[1,2]$ for $12$ steps. Compare the last midpoint to Newton’s $x_5$ from Task 2.

**3c.** One sentence: name one advantage of bisection over Newton, and one advantage of Newton over bisection.


In [ ]:
# Task 3 — complete the TODOs. This cell fails until you do.

def bisection(f, a, b, n_steps=12, tol=1e-14):
    """Halve a sign-change bracket. Returns (midpoints, final (a, b))."""
    a, b = float(a), float(b)
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError("f(a) and f(b) must have opposite signs")
    mids = []
    for _ in range(n_steps):
        m = (a + b) / 2  # TODO: (a + b) / 2
        if m is None:
            raise NotImplementedError("Task 3: set the midpoint m")
        mids.append(m)
        fm = f(m)
        if abs(fm) < tol:
            break
        # TODO: keep the half that still has a sign change.
        # if fa * fm < 0:  b, fb = m, fm
        # else:            a, fa = m, fm
        if fa * fm < 0:
          b, fb = m, fm
        else:
          a, fa = m, fm
    return np.array(mids), (a, b)


f_cubic = lambda t: 3 * t**3 + 2 * t - 7 * np.pi
print("3a  f(1) =", f_cubic(1.0), " f(2) =", f_cubic(2.0))

mids, bracket = bisection(f_cubic, 1.0, 2.0, n_steps=12)
print("3b  last midpoint =", mids[-1], " bracket =", bracket)
print("     |f(last)|    =", abs(f_cubic(mids[-1])))


3a  f(1) = -16.991148575128552  f(2) = 6.008851424871448
3b  last midpoint = 1.828369140625  bracket = (1.828125, 1.828369140625)
     |f(last)|    = 0.0019402337571072792


**Your answers (Task 3)**

- 3a: f(1) = −16.991 and f(2) = 6.009, so the values have opposite signs. Since f is continuous, there must be at least one root between 1 and 2.
- 3b: After 12 steps, the bisection method gives a midpoint of about 1.8284. Newton's method gives x₅ ≈ 1.8283. Both methods give very similar results and find the root at approximately 1.828.
- 3c: Bisection is more reliable than Newton's method because it is guaranteed to converge if there is a sign change and the function is continuous. It also doesn't require a derivative. Newton's method usually converges much faster, needing fewer iterations to get close to the root.


## Task 4 — Critical points and 1D gradient descent (questions)

Cohen: a **c.p.** is an $x$ where $f'=0$ or $f'$ is **d.n.e.**, and $f$ itself is defined (§6.1). 1D GD (6.8):

$$x\leftarrow x-\eta f'(x)$$

$\eta$ is the **learning rate**, not a weight.

This task is the **linear rehearsal** for graded Task 9. Task 9 repeats the same three updates on Lab 2’s logistic neuron. Here the prediction is still Lab 1’s line $\hat{y}=wx$.

**4a.** For $f(x)=x^{3}-x^{2}+3$ (Cohen (6.1)), find all real $x$ with $f'(x)=0$. Use `sp.diff` / `sp.solve`, and write the c.p.s.

**4b.** Lab 1 cost: $x=3$, $y=2$, $\hat{y}=wx$, $C(w)=(wx-y)^{2}$, start at $w=1$. Write $C$ and $C'$ (chain rule: $C'(w)=2(wx-y)x$). Compute $C'(1)$. In which direction do you nudge $w$? Take **one** GD step with $\eta=0.01$ using (6.8).

**4c.** Newton on $g=C'$ finds c.p.s of $C$. From $w=1$, take one Newton step on $C'$ (you need $C''$). Compare to the GD step and to the exact root of $C'(w)=0$.

**4d.** From the same $w=1$, take one Newton step on $C$ itself (hunt $C=0$). Write both next weights. One sentence: which problem is each Newton step solving? (Lecture 3; this is Task 9c on a simpler $C$.)

**4e.** Reuse Task 3’s `bisection` on **$C'$** (not $C$) over $[0,1]$ for $12$ steps. Why does a sign change of $C'$ locate a c.p. of $C$?


In [ ]:
# Task 4 — complete the TODOs. This cell fails until you do.
# Reuses newton_step (Task 2) and bisection (Task 3) if you wrote them.

# 4a — Cohen (6.1)
f_sym = x**3 - x**2 + 3 # TODO: x**3 - x**2 + 3
if f_sym is None:
    raise NotImplementedError("Task 4a: set f_sym")
fp_sym = sp.diff(f_sym, x)
cps = sp.solve(fp_sym, x)  # TODO: sp.solve(fp_sym, x)
if cps is None:
    raise NotImplementedError("Task 4a: solve f'=0")
print("4a  f' =", fp_sym)
print("    solve f'=0:", cps)

# 4b–4e  Lab 1 numbers (linear ŷ = w x). Task 9 uses logistic ŷ = σ(w x).
x_data, y_actual, w0, eta = 3.0, 2.0, 1.0, 0.01


def C(w, x=x_data, y=y_actual):
    return (w * x - y)**2 # TODO: return (w * x - y)**2


def Cp(w, x=x_data, y=y_actual):
    return 2 * (w * x - y) * x # TODO: chain rule 2 * (w * x - y) * x

def Cpp(w, x=x_data, y=y_actual):
    return 2 * x**2 # TODO: 2 * x**2


print("4b  C'(1) =", Cp(w0))

w_gd =  w0 - eta * Cp(w0)  # TODO: w0 - eta * Cp(w0)
if w_gd is None:
    raise NotImplementedError("Task 4b: one GD step (6.8)")
print("    one GD step =", w_gd)

w_newton_Cp = w0 - Cp(w0) / Cpp(w0)  # TODO: w0 - Cp(w0) / Cpp(w0)
w_newton_C = w0 - C(w0) / Cp(w0)   # TODO: w0 - C(w0) / Cp(w0)
if w_newton_Cp is None or w_newton_C is None:
    raise NotImplementedError("Task 4c/4d: both Newton updates")
print("4c  Newton on C' =", w_newton_Cp)
print("    exact C'=0  -> w =", y_actual / x_data)
print("4d  Newton on C  =", w_newton_C)

mids_cp, bracket_cp = bisection(Cp, 0.0, 1.0, n_steps=12)
print("4e  last midpoint of C' =", mids_cp[-1], " bracket =", bracket_cp)


4a  f' = 3*x**2 - 2*x
    solve f'=0: [0, 2/3]
4b  C'(1) = 6.0
    one GD step = 0.94
4c  Newton on C' = 0.6666666666666667
    exact C'=0  -> w = 0.6666666666666666
4d  Newton on C  = 0.8333333333333334
4e  last midpoint of C' = 0.666748046875  bracket = (0.66650390625, 0.666748046875)


**Your answers (Task 4)**

- 4a:

f'(x) = 3x² − 2x

Setting f'(x) = 0 gives:

x(3x − 2) = 0

So the critical points are x = 0 and x = 2/3.

- 4b:

C(w) = (3w − 2)²

C'(w) = 6(3w − 2) = 18w − 12

C'(1) = 6, which is positive, so w should decrease.

Using gradient descent with a learning rate of 0.01:

w = 1 − 0.01(6) = 0.94

- 4c:

Using Newton's method:

w = 1 − C'(1)/C''(1) = 1 − 6/18 = 2/3

This gives the exact minimum. Gradient descent with a learning rate of 0.01 only reaches 0.94 after one step, so it takes longer to get to the minimum.

- 4d:

Using Newton's method on C:

w = 1 − C(1)/C'(1) = 1 − 1/6 = 5/6 ≈ 0.8333

Newton's method on C' finds the minimum by solving C'(w) = 0, while Newton's method on C finds where the cost equals zero by solving C(w) = 0. These are two different problems.

- 4e:

C'(0) = −12 < 0

C'(1) = 6 > 0

Since the derivative changes sign from negative to positive, there is a minimum between 0 and 1. Bisection finds the root of C', which is the critical point of C. The last midpoint should be close to 2/3.


## Task 5 — Multivariable limits (questions)

Cohen §10.2. If two paths to $(x_0,y_0)$ give different values, the limit is **d.n.e.**

$$f(x,y)=\frac{xy}{x^{2}+y^{2}}\qquad\text{(10.8)}$$

(undefined at $(0,0)$).

**5a.** Limit of $f$ as $(x,y)\to(0,0)$ along $y=0$; along $x=0$; along $x=y$. Numbers or **d.n.e.** for each path.

**5b.** Does $\lim_{(x,y)\to(0,0)}f(x,y)$ exist? Write a number or **d.n.e.** and one sentence why.

**5c.** Repeat 5b for $h(x,y)=2x+4/y$ as $(x,y)\to(0,2)$ (Cohen’s plug-in example around (10.13)). Does this one exist?


In [ ]:
# Task 5 — complete the TODOs. This cell fails until you do.
# Task 9e is this law with weights (w1, w2) in place of (x, y).

def f_path(xx, yy):
    return xx * yy / (xx**2 + yy**2) # TODO: return xx * yy / (xx**2 + yy**2)    Cohen (10.8)

hs = np.array([1e-1, 1e-2, 1e-3, 1e-4])
along_y0 = [f_path(h, 0.0) for h in hs]  # TODO: [f_path(h, 0.0) for h in hs]
along_x0 = [f_path(0.0, h) for h in hs]  # TODO: [f_path(0.0, h) for h in hs]
along_xy = [f_path(h, h) for h in hs]  # TODO: [f_path(h, h) for h in hs]
if along_y0 is None or along_x0 is None or along_xy is None:
    raise NotImplementedError("Task 5a: evaluate the three paths")
print("along y=0:", along_y0)
print("along x=0:", along_x0)
print("along x=y:", along_xy)

# 5c — Cohen around (10.13). SymPy cannot do a 2D limit in one call.
x_s, y_s = sp.symbols("x y", real=True)
h = 2 * x_s + 4 / y_s  # TODO: 2 * x_s + 4 / y_s
if h is None:
    raise NotImplementedError("Task 5c: set h")
print("5c sympy limit x then y at (0,2):", sp.limit(sp.limit(h, x_s, 0), y_s, 2))
print("         limit y then x at (0,2):", sp.limit(sp.limit(h, y_s, 2), x_s, 0))


along y=0: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
along x=0: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
along x=y: [np.float64(0.5), np.float64(0.5), np.float64(0.5), np.float64(0.5)]
5c sympy limit x then y at (0,2): 2
         limit y then x at (0,2): 2


**Your answers (Task 5)**

- 5a:

Along y = 0, the limit is 0.

Along x = 0, the limit is also 0.

Along x = y, the limit is 1/2.
- 5b:

The limit does not exist because different paths give different values. Along x = y, the limit is 1/2, while along y = 0 and x = 0, the limit is 0. Since these values don't agree, the limit does not exist (d.n.e.).

- 5c:

Substituting (0, 2) gives 2(0) + 4/2 = 2.

The function is continuous at (0, 2) because y = 2 is not zero. Therefore, the limit exists and equals 2.


## Task 6 — Partial derivatives and Clairaut (questions)

Cohen §10.3–10.4. Treat the other variable as constant.

$$f_x=\lim_{h\to 0}\frac{f(x+h,y)-f(x,y)}{h},\qquad
f_y=\lim_{h\to 0}\frac{f(x,y+h)-f(x,y)}{h}\qquad\text{(10.24)--(10.25)}$$

**6a.** By hand, for $f(x,y)=x^{2}+xy+y$ (10.18), write $f_x$ and $f_y$. Confirm with `sp.diff`.

**6b.** Mixed partials: $f_{xy}=(f_x)_y$ and $f_{yx}=(f_y)_x$ (10.28)–(10.29). For
$$u(x,y)=\cos(2xy^{2})+x^{2}-ye^{x}\qquad\text{(10.36)}$$
compute $u_{xy}$ and $u_{yx}$ with SymPy. Are they equal? Name the theorem.

**6c.** When does Cohen say mixed partials might **fail** to agree?


In [ ]:
# Task 6 — complete the TODOs. This cell fails until you do.

f = x**2 + x*y + y  # TODO: x**2 + x * y + y          Cohen (10.18)
if f is None:
    raise NotImplementedError("Task 6a: set f")
fx = sp.diff(f, x)  # TODO: sp.diff(f, x)
fy = sp.diff(f, y)  # TODO: sp.diff(f, y)
if fx is None or fy is None:
    raise NotImplementedError("Task 6a: differentiate")
print("6a  f_x =", fx)
print("    f_y =", fy)

u = sp.cos(2*x*y**2) + x**2 - y*sp.exp(x)  # TODO: sp.cos(2 * x * y**2) + x**2 - y * sp.exp(x)   (10.36)
if u is None:
    raise NotImplementedError("Task 6b: set u")
uxy = sp.diff(u, x, y)  # TODO: sp.diff(u, x, y)
uyx = sp.diff(u, y, x)  # TODO: sp.diff(u, y, x)
if uxy is None or uyx is None:
    raise NotImplementedError("Task 6b: mixed partials")
print("6b  u_xy =", sp.simplify(uxy))
print("    u_yx =", sp.simplify(uyx))
print("    equal?", sp.simplify(uxy - uyx) == 0)


6a  f_x = 2*x + y
    f_y = x + 1
6b  u_xy = -8*x*y**3*cos(2*x*y**2) - 4*y*sin(2*x*y**2) - exp(x)
    u_yx = -8*x*y**3*cos(2*x*y**2) - 4*y*sin(2*x*y**2) - exp(x)
    equal? True


**Your answers (Task 6)**

- 6a:

fₓ = 2x + y

fᵧ = x + 1
- 6b:

uₓᵧ = −4y sin(2xy²) − 8xy³ cos(2xy²) − eˣ

uᵧₓ = −4y sin(2xy²) − 8xy³ cos(2xy²) − eˣ

Since both mixed partial derivatives are equal, this agrees with Clairaut's theorem.

- 6c:

Mixed partial derivatives may not be equal if the relevant partial derivatives are not continuous near the point.


## Task 7 — Gradient, Hessian, 2D gradient descent (questions)

Cohen (10.43), (10.54), (10.63):

$$\nabla f=\begin{bmatrix}f_x\\ f_y\end{bmatrix},\qquad
z_0\leftarrow z_0-\eta\nabla f(z_0),\qquad z_0=(x_0,y_0).$$

Hessian $H_f$ organises the four second partials. Cohen’s subscript layout is
$$H_f=\begin{bmatrix}f_{xx}&f_{yx}\\ f_{xy}&f_{yy}\end{bmatrix}\qquad\text{(10.54)}.$$
When Clairaut holds, $f_{xy}=f_{yx}$ and this matches the usual symmetric matrix.

Let $f(x,y)=x^{2}+y^{2}$.

**7a.** Write $\nabla f$ and $H_f$ by hand. Confirm with SymPy (`sp.diff` / `sp.hessian`).

**7b.** One GD step from $z_0=(1,1)$ with $\eta=0.1$. Did $f$ decrease?

**7c.** At the origin, what is $\nabla f$? Is the origin a c.p. of this $f$? Min, max, or neither — and how can you tell from the formula (no $D$-test required)?


In [ ]:
# Task 7 — complete the TODOs. This cell fails until you do.

f2 = x**2 + y**2  # TODO: x**2 + y**2
if f2 is None:
    raise NotImplementedError("Task 7a: set f2")
grad = sp.Matrix([sp.diff(f2, x), sp.diff(f2, y)])  # TODO: sp.Matrix([sp.diff(f2, x), sp.diff(f2, y)])
H = sp.hessian(f2, [x, y])     # TODO: sp.hessian(f2, [x, y])
if grad is None or H is None:
    raise NotImplementedError("Task 7a: set grad and H")
print("7a  grad =")
sp.pprint(grad)
print("    Hessian =")
sp.pprint(H)

z0 = np.array([1.0, 1.0])
eta = 0.1
g_num = np.array([2 * z0[0], 2 * z0[1]], dtype=float)  # TODO: np.array([2 * z0[0], 2 * z0[1]], dtype=float)
z1 = z0 - eta * g_num    # TODO: z0 - eta * g_num    (10.63)
if g_num is None or z1 is None:
    raise NotImplementedError("Task 7b: one 2D GD step")
f_of = lambda z: z[0] ** 2 + z[1] ** 2
print("7b  z1 =", z1, " f(z0) =", f_of(z0), " f(z1) =", f_of(z1))


7a  grad =
⎡2⋅x⎤
⎢   ⎥
⎣2⋅y⎦
    Hessian =
⎡2  0⎤
⎢    ⎥
⎣0  2⎦
7b  z1 = [0.8 0.8]  f(z0) = 2.0  f(z1) = 1.2800000000000002


**Your answers (Task 7)**

- 7a:

∇f = [2x, 2y]ᵀ

H_f = [[2, 0], [0, 2]]

- 7b:

The gradient at (1, 1) is (2, 2). Using a learning rate of 0.1, the next point is (1, 1) − 0.1(2, 2) = (0.8, 0.8). The function value decreases from f(1, 1) = 2 to f(0.8, 0.8) = 1.28, so the function value has decreased after the gradient descent step.

- 7c:

The gradient at (0, 0) is (0, 0), so the origin is a critical point. It is a minimum because f(x, y) = x² + y² is always greater than or equal to 0. At the origin, f(0, 0) = 0, which is the smallest possible value of the function.


## Task 8 — Model: two weights (questions)

Keep Lab 1–2 language. The prediction is $\hat{y}=w_1 x+w_2$. Here $w_1$ is a **weight** and $w_2$ is an intercept (still a parameter, not the prediction). Cost on two data points $(x_i,y_i)$:

$$C(w_1,w_2)=\sum_i\bigl(w_1 x_i+w_2-y_i\bigr)^{2}.$$

Use the Lab 1 point $(x,y)=(3,2)$ and a second point $(1,0)$. Start at $(w_1,w_2)=(1,0)$.

**8a.** Write $C$ and $\nabla C=\begin{bmatrix}C_{w_1}\\ C_{w_2}\end{bmatrix}$ in the next cell. Evaluate $\nabla C(1,0)$.

**8b.** One 2D GD step (10.63) with $\eta=0.01$. Did $C$ decrease?

**8c.** Why would a *single* data point make $\nabla C=\mathbf{0}$ a whole line of solutions rather than one pair $(w_1,w_2)$?

**8d.** One sentence: why is the update in 8b gradient descent rather than Newton's method? In more than one variable, what equation would Newton-based optimization try to solve?

**Prep for Task 9.** Graded Task 9 keeps one weight but changes the prediction to $\hat{y}=\sigma(wx)$ (Lab 2). You will again write $C$ and $C'$, take one GD step (6.8), compare Newton on $C'$ with Newton on $C$, and bisect $C'$. Finish Tasks 2–4 before you start it.


In [ ]:
# Task 8 — complete the TODOs. This cell fails until you do.

pts = [(3.0, 2.0), (1.0, 0.0)]


def C(w1, w2, data=pts):
    return sum((w1 * xi + w2 - yi)**2 for xi, yi in data) # TODO: sum of (w1 * xi + w2 - yi)**2


def grad_C(w1, w2, data=pts):
    # TODO: return np.array([dC/dw1, dC/dw2])
    # dC/dw1 = sum 2*(w1*xi + w2 - yi)*xi
    # dC/dw2 = sum 2*(w1*xi + w2 - yi)
    d1 = sum(2 * (w1 * xi + w2 - yi) * xi for xi, yi in data)
    d2 = sum(2 * (w1 * xi + w2 - yi) for xi, yi in data)
    return np.array([d1, d2], dtype=float)


w0 = np.array([1.0, 0.0])
g0 = grad_C(*w0)
eta = 0.01
w_next = w0 - eta * g0 # TODO: w0 - eta * g0    (10.63)
if w_next is None:
    raise NotImplementedError("Task 8b: one 2D GD step")
print("8a  grad C(1,0) =", g0)
print("8b  next z      =", w_next)
print("    C before    =", C(*w0), " C after =", C(*w_next))

C_sym = (w1 * 3 + w2 - 2)**2 + (w1 * 1 + w2 - 0)**2  # TODO: (w1 * 3 + w2 - 2)**2 + (w1 * 1 + w2 - 0)**2
if C_sym is None:
    raise NotImplementedError("Task 8: set C_sym")
grad_sym = sp.Matrix([sp.diff(C_sym, w1), sp.diff(C_sym, w2)])
print("sympy grad at (1,0):", grad_sym.subs({w1: 1, w2: 0}))


8a  grad C(1,0) = [8. 4.]
8b  next z      = [ 0.92 -0.04]
    C before    = 2.0  C after = 1.2928000000000002
sympy grad at (1,0): Matrix([[8], [4]])


**Your answers (Task 8)**

- 8a:

The partial derivatives are C_w₁ = 20w₁ + 8w₂ − 12 and C_w₂ = 8w₁ + 4w₂ − 4. Evaluating them at (1, 0) gives the gradient ∇C(1, 0) = (8, 4).

- 8b:

Using a learning rate of 0.01, the next point is (1, 0) − 0.01(8, 4) = (0.92, −0.04). The cost decreases from C(1, 0) = 2 to C(0.92, −0.04) = 1.2928, so the update reduces the cost.

- 8c:

With only one data point, the equation is w₁x + w₂ = y, which represents a line in the (w₁, w₂) plane. This means there are infinitely many combinations of w₁ and w₂ that give zero cost.

- 8d:

Gradient descent updates the weights by moving in the opposite direction of the gradient to reduce the cost. Newton-based optimisation uses the gradient and the Hessian matrix to solve ∇C = 0 and find a minimum.


### Solutions (tutor) — Task 8

**8a.** $C_{w_1}=2(3w_1+w_2-2)\cdot 3+2(w_1+w_2)\cdot 1$, $C_{w_2}=2(3w_1+w_2-2)+2(w_1+w_2)$. At $(1,0)$: $\nabla C=(8,4)$.

**8b.** $z\leftarrow(1,0)-0.01(8,4)=(0.92,-0.04)$. $C(1,0)=2$, $C$ after $\approx 1.293$: decrease.

**8c.** One point $(x,y)$ only constrains $w_1 x+w_2=y$, a line in the $(w_1,w_2)$-plane. Infinitely many parameter pairs give zero cost.

**8d.** Gradient descent moves opposite to the gradient. Newton-based optimization seeks a solution of $\nabla C=\mathbf 0$, using the Hessian in the multivariable Newton update. In one dimension this reduces to applying Newton's method to $C'(w)=0$.


In [ ]:
# Solutions (tutor) — Task 8 complete code

pts = [(3.0, 2.0), (1.0, 0.0)]


def C(w1, w2, data=pts):
    return sum((w1 * xi + w2 - yi) ** 2 for xi, yi in data)


def grad_C(w1, w2, data=pts):
    d1 = sum(2 * (w1 * xi + w2 - yi) * xi for xi, yi in data)
    d2 = sum(2 * (w1 * xi + w2 - yi) for xi, yi in data)
    return np.array([d1, d2], dtype=float)


w0 = np.array([1.0, 0.0])
g0 = grad_C(*w0)
eta = 0.01
w_next = w0 - eta * g0
print("8a  grad C(1,0) =", g0)
print("8b  next z      =", w_next)
print("    C before    =", C(*w0), " C after =", C(*w_next))

C_sym = (w1 * 3 + w2 - 2) ** 2 + (w1 * 1 + w2 - 0) ** 2
grad_sym = sp.Matrix([sp.diff(C_sym, w1), sp.diff(C_sym, w2)])
print("sympy grad at (1,0):", grad_sym.subs({w1: 1, w2: 0}))


8a  grad C(1,0) = [8. 4.]
8b  next z      = [ 0.92 -0.04]
    C before    = 2.0  C after = 1.2928000000000002
sympy grad at (1,0): Matrix([[8], [4]])


## Task 9 — For credit: logistic neuron

**This task is for credit (20 marks).** You must write the code (the next cell fails until every `TODO` is done) and the written answers.

Laws and notation follow Cohen and Lectures 1–3. **d.n.e.** means **d**oes **n**ot **e**xist (§4.5, §10.2). Never write an undefined “DNE”. A **c.p.** is a **c**ritical **p**oint: $f'=0$ or $f'$ is **d.n.e.**, and $f$ itself is defined (§6.1). $\eta$ is a **learning rate**, not a weight.

**Model (Lab 2 Part C, not Lab 1's line).** One input $x=3$, target $y=0.8$ (a probability). Prediction and cost:

$$\hat{y}=\sigma(wx),\qquad \sigma(u)=\frac{1}{1+e^{-u}},\qquad C(w)=\bigl(\sigma(wx)-y\bigr)^{2}.$$

$w$ is a **weight**. $\hat{y}$ is the **prediction**. $C$ is the **cost**. Start at $w=0$. Cohen’s algorithms (modified from (6.8), (9.23), and the cubic in (9.22) / Task 2–3):

$$w\leftarrow w-\eta C'(w)\qquad\text{(GD; (6.8))},\qquad
w\leftarrow w-\frac{g(w)}{g'(w)}\qquad\text{(Newton (9.23) on a chosen }g\text{)}.$$

**9a (5 marks).** Implement `sigma`, `C`, and `C_prime`. Use the chain rule (Lecture 2 / Lab 2):

$$C'(w)=2\bigl(\sigma(wx)-y\bigr)\,\sigma(wx)\bigl(1-\sigma(wx)\bigr)\,x.$$

Print $C(0)$ and $C'(0)$. One sentence: do you nudge $w$ **up** or **down**, and why? (Sign of $C'$, Labs 1–3.)

**9b (3 marks).** One GD step (6.8) from $w=0$ with $\eta=0.5$. Print the new weight and $C$ after the step. Did $C$ decrease?

**9c (4 marks).** Two different Newton steps from $w=0$:

- Newton on $g=C'$ (find a **c.p.** of $C$). You need $C''(0)$; build $C$ in SymPy and differentiate twice.
- Newton on $g=C$ itself (hunt a root $C=0$).

Write both next weights. One sentence: which of these is the optimisation tool from Lecture 3, and which is solving a *different* problem?

**9d (3 marks).** $C'$ changes sign on $[0,1]$ (check). Bisect **$C'$** (not $C$) on $[0,1]$ for $12$ steps — reuse Task 3's `bisection` or write a short loop. Compare the last midpoint to $\ln\bigl(y/(1-y)\bigr)/x$ (the exact weight that makes $\hat{y}=y$). One sentence: why does a sign change of $C'$ locate a c.p. of $C$?

**9e (2 marks).**

- When is the Newton update on $C'$ **d.n.e.**? (Cohen §9.2.2.)
- Modified Cohen (10.8), same law, Lab 3 weight names:
  $$g(w_1,w_2)=\frac{w_1 w_2}{w_1^{2}+w_2^{2}}$$
  (undefined at $(0,0)$). Limit as $(w_1,w_2)\to(0,0)$ along $w_2=0$, and along $w_1=w_2$. Does $\lim_{(w_1,w_2)\to(0,0)}g$ exist? Write a number or **d.n.e.** for each, and one sentence why.

**9f (3 marks).** Three sentences:

1. What is free to change in this model, and why is the target $y=0.8$ rather than Lab 1's $y=2$?
2. Newton and GD both subtract a derivative-related term. What *problem* is each solving here?
3. Name one advantage of bisection on $C'$ over Newton on $C'$, and one advantage of Newton.


In [ ]:
# Task 9 — For credit. Complete every TODO.
# Running this cell as-is will fail (NotImplementedError).

x_data = 3.0
y_prob = 0.8  # Lab 2: target in (0, 1) because sigma only outputs (0, 1)
w0 = 0.0
eta = 0.5


def sigma(u):
     return 1 / (1 + np.exp(-u)) # TODO: return 1 / (1 + exp(-u))


def C(w, x=x_data, y=y_prob):
    # TODO: yhat = sigma(w * x); return (yhat - y)**2
    yhat = sigma(w * x)
    return (yhat - y)**2



def C_prime(w, x=x_data, y=y_prob):
    return 2 * (sigma(w*x) - y) * sigma(w*x) * (1 - sigma(w*x)) * x
    # TODO: 2 * (sigma(w*x) - y) * sigma(w*x) * (1 - sigma(w*x)) * x



# 9a
print("9a  C(0)  =", C(w0))
print("    C'(0) =", C_prime(w0))

# 9b
w_gd = w0 - eta * C_prime(w0)  # TODO: w0 - eta * C_prime(w0)
if w_gd is None:
    raise NotImplementedError("Task 9b: one GD step (6.8)")
print("9b  w after GD =", w_gd, "  C after =", C(w_gd))

# 9c — C''(w0) via sympy (Cohen: analytic derivative, then plug in)
w_sym = sp.symbols("w", real=True)
C_sym = (1/(1 + sp.exp(-w_sym * x_data)) - y_prob)**2 # TODO: (1/(1 + exp(-w_sym * x_data)) - y_prob)**2
if C_sym is None:
    raise NotImplementedError("Task 9c: set C_sym")
Cp_sym = sp.diff(C_sym, w_sym)
Cpp_sym = sp.diff(Cp_sym, w_sym)
Cpp0 = float(Cpp_sym.subs(w_sym, w0))
w_newton_on_Cp = w0 - C_prime(w0) / Cpp0   # TODO: w0 - C_prime(w0) / Cpp0     (Newton on C')
w_newton_on_C = w0 - C(w0) / C_prime(w0)   # TODO: w0 - C(w0) / C_prime(w0)    (Newton on C)
if w_newton_on_Cp is None or w_newton_on_C is None:
    raise NotImplementedError("Task 9c: both Newton updates")
print("9c  C''(0)           =", Cpp0)
print("    Newton on C'     =", w_newton_on_Cp)
print("    Newton on C      =", w_newton_on_C)

# 9d — bisect C' on [0, 1]
# Reuse bisection from Task 3 if it is defined; otherwise write a short loop.
try:
    mids_9, bracket_9 = bisection(C_prime, 0.0, 1.0, n_steps=12)
except NameError:
    mids_9 = None  # TODO: 12 midpoints from bisecting C_prime on [0, 1]
    bracket_9 = None
    if mids_9 is None:
        raise NotImplementedError("Task 9d: bisect C' (define bisection or a loop)")
w_exact = float(np.log(y_prob / (1.0 - y_prob)) / x_data)
print("9d  last midpoint =", float(mids_9[-1]), " bracket =", bracket_9)
print("    exact w*      =", w_exact, "  |f last| on C' =", abs(C_prime(float(mids_9[-1]))))


9a  C(0)  = 0.09000000000000002
    C'(0) = -0.45000000000000007
9b  w after GD = 0.22500000000000003   C after = 0.01887275962252334
9c  C''(0)           = 1.125
    Newton on C'     = 0.4000000000000001
    Newton on C      = 0.2
9d  last midpoint = 0.462158203125  bracket = (0.4619140625, 0.462158203125)
    exact w*      = 0.46209812037329695   |f last| on C' = 2.7681640831413177e-05


**Your answers (Task 9 — for credit)**

- 9a $C(0)$, $C'(0)$, nudge:

At w = 0, σ(0) = 0.5, so C(0) = (0.5 − 0.8)² = 0.09. The derivative is C′(0) = 2(0.5 − 0.8)(0.5)(0.5)(3) = −0.45. Since C′(0) is negative, increasing w slightly will reduce the cost, so w should be nudged upward.

- 9b new $w$, $C$ after, decrease? :

Using gradient descent with a learning rate of η = 0.5, the new weight is w = 0 − 0.5(−0.45) = 0.225. The cost at this point is C(0.225) ≈ 0.0189, which is lower than the original cost of 0.09. Therefore, the cost decreased.

- 9c Newton on $C'$, Newton on $C$, which problem each solves:

Applying Newton's method to C′ gives w = 0 − C′(0)/C″(0) = 0 − (−0.45)/1.125 = 0.4. This solves C′(w) = 0, which helps find a stationary point of the cost function.

Applying Newton's method to C gives w = 0 − C(0)/C′(0) = 0 − 0.09/(−0.45) = 0.2. This solves C(w) = 0, finding a point where the cost is zero. The two methods solve different equations: Newton's method on C′ seeks a stationary point, while Newton's method on C seeks a root of the cost function.

- 9d last midpoint vs $w^{*}$; why a sign change of $C'$ is a c.p. of $C$:

C′(0) = −0.45 < 0 and C′(1) ≈ 0.041 > 0, so C′ changes sign between 0 and 1. After 12 bisection steps, the last midpoint is approximately 0.4621. The exact value is w* = ln(4)/3 ≈ 0.4621, so the two values match. A sign change from negative to positive means C′ crosses zero, indicating a critical point of C. Since the derivative changes from negative to positive, this point is a minimum.

- 9e Newton on $C'$ is **d.n.e.** when:

Newton's method on C′ is d.n.e. when C″(w) = 0, because the update divides by C″(w). Division by zero makes the update undefined.

- 9e path $w_2=0$; path $w_1=w_2$; 2D limit:

Along the path w₂ = 0, g(w₁, 0) = 0. Along the path w₁ = w₂, g(w₁, w₁) = 1/2. Since the two paths give different values, the 2D limit d.n.e.
- 9f.1:

Only w can be adjusted. The target is y = 0.8 instead of y = 2 because the sigmoid function only produces values between 0 and 1, so it cannot reach a target of 2.

- 9f.2:

Newton's method on C′ solves C′(w) = 0 to find a stationary point, such as a minimum. Newton's method on C solves C(w) = 0 to find where the cost is zero. Gradient descent minimises C by updating w in the opposite direction of the derivative.

- 9f.3:

Bisection on C′ is guaranteed to find a root within the interval if C′ is continuous and changes sign at the endpoints, but it can be slow. Newton's method on C′ is usually faster, but it requires C″ and can fail if C″(w) = 0.


## Submit

**Submit (myeLearning):** Tasks 0–8 are the in-lab work. **Task 9 is for credit (20 marks)** — complete the templated code and the written answers. Use **d.n.e.** when a limit or a Newton update fails to exist. Never write “DNE”. Task 8 is the in-lab modelling closer; Task 9 is the assessed logistic / algorithm task.

**Discussion forum (RISE):** Newton and GD both subtract a derivative-related term. What *problem* is each solving?

**Tutor:** delete or hide every cell whose heading starts with **Solutions (tutor)** or **Mark scheme (tutor)** before distributing.

**Next week:** integration (syllabus 3.1–3.4).
